IMPORTANTE: Usar GPU A100

In [1]:
!pip install -q pandas==2.2.2 chromadb sentence-transformers ebooklib beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
# 1) Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 2) Configuración de rutas persistentes
from pathlib import Path
MODEL_NAME = 'BAAI/bge-m3'
PERSIST_BASE = Path('/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base')
CHROMA_DIR = PERSIST_BASE / 'chroma_db'
CHUNKS_DIR = PERSIST_BASE / 'chunks'
for p in [PERSIST_BASE, CHROMA_DIR, CHUNKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# BBDD 1: INGESTIÓN DEL LIBRO

Ejecución una sola vez

In [ ]:
# 3) Clonar/actualizar repositorio dentro de Drive
# IMPORTANTE: cambia REPO_URL por tu remoto real si hace falta.
REPO_URL = 'https://github.com/javierrcastroo/LLM-Juego-de-Tronos.git'

!git clone {REPO_URL}

Cloning into 'LLM-Juego-de-Tronos'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 95 (delta 40), reused 19 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 4.98 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (40/40), done.


Libro de Texto 1 de GitHub para dentro de la Chroma + parquet.

El parquet es una copia de seguridad

In [ ]:
import os
import re
import uuid
import torch
import gc
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import ebooklib
from ebooklib import epub
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

PERSIST_BASE = Path('/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base')
CHROMA_DIR = PERSIST_BASE / 'chroma_db'
CHUNKS_DIR = PERSIST_BASE / 'chunks'
REPO_DIR = PERSIST_BASE / 'LLM-Juego-de-Tronos'

COLLECTION_NAME = 'got_book1_chunks'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
BATCH_SIZE = 64

for p in [CHROMA_DIR, CHUNKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def clean_text_safe(text: str) -> str:
    """Limpia HTML y elimina caracteres que rompen los kernels de la GPU."""
    text = re.sub(r'\s+', ' ', text)
    text = text.encode("ascii", "ignore").decode()
    return text.strip()

def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk) > 60:
            chunks.append(chunk)
        if i + chunk_size >= len(text): break
    return chunks

def load_epub_clean(epub_path):
    book = epub.read_epub(str(epub_path))
    sections = []
    chapter_idx = 0
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        html = item.get_content()
        soup = BeautifulSoup(html, 'html.parser')
        text = clean_text_safe(soup.get_text(' ', strip=True))
        if len(text) < 200: continue
        chapter_idx += 1
        sections.append({
            'chapter_index': chapter_idx,
            'chapter_id': item.get_id(),
            'chapter_name': item.get_name(),
            'text': text,
        })
    return sections

📖 Cargando libro: /content/LLM-Juego-de-Tronos/Canci_243_n_de_Hielo_y_Fuego_01_-_Juego_de_Tronos.epub
✅ 2028 chunks listos.
🧠 Cargando modelo estable: BAAI/bge-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🚀 Iniciando ingesta en ChromaDB...
📦 Procesados: 0/2028 chunks...
📦 Procesados: 320/2028 chunks...
📦 Procesados: 640/2028 chunks...
📦 Procesados: 960/2028 chunks...
📦 Procesados: 1280/2028 chunks...
📦 Procesados: 1600/2028 chunks...
📦 Procesados: 1920/2028 chunks...
🏁 ¡CONSEGUIDO! Total en Chroma: 2028

🔍 PRUEBA DE BÚSQUEDA:
[1] ntos saba Jon si el maestre se haba quedado porque era dbil y cobarde, o porque era fuerte y honorable. Pero comprenda lo que le haba contado sobre el dolor de elegir. Lo comprenda demasiado bien. Tyr...
[2] Jon Nieve. El chico absorbi la informacin en silencio. No tena el apellido de los Stark, pero s el rostro: alargado, solemne, cauteloso, un rostro que no delataba nada. Fuera quien fuera su madre, no ...
[3] JON (1) Haba ocasiones, aunque no muchas, en las que Jon Nieve se alegraba de ser el hijo bastardo. Aquella noche, mientras se llenaba una vez ms la copa de vino de la jarra de un mozo que pasaba junt...


In [ ]:
EPUB_PATH = '/content/LLM-Juego-de-Tronos/Canci_243_n_de_Hielo_y_Fuego_01_-_Juego_de_Tronos.epub'

print(f"📖 Cargando libro: {EPUB_PATH}")
sections = load_epub_clean(EPUB_PATH)
records = []
for sec in sections:
    c_chunks = chunk_text(sec['text'])
    for j, ch in enumerate(c_chunks):
        records.append({
            'chunk_id': f"{sec['chapter_id']}_{j}",
            'chapter_index': sec['chapter_index'],
            'chapter_name': sec['chapter_name'],
            'chunk_index': j,
            'text': ch,
        })

chunks_df = pd.DataFrame(records)
chunks_df.to_parquet(CHUNKS_DIR / 'chunks.parquet', index=False)
print(f"✅ {len(chunks_df)} chunks listos.")

gc.collect()
# No llamamos a empty_cache() aquí para evitar errores si la sesión no se reinició bien
print(f"🧠 Cargando modelo estable: {MODEL_NAME}")
embedder = SentenceTransformer(MODEL_NAME, device="cuda")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Limpiamos colección previa para asegurar consistencia
if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"} # BGE funciona mejor con similitud coseno
)

print(f"🚀 Iniciando ingesta en ChromaDB...")
texts = chunks_df['text'].tolist()
metadatas = chunks_df.drop(columns=['text']).to_dict(orient='records')
ids = [str(uuid.uuid4()) for _ in texts]

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_meta = metadatas[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]

    # Inferencia sin gradientes para ahorrar memoria y evitar asserts
    with torch.no_grad():
        embeddings = embedder.encode(
            batch_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True
        ).tolist()

    collection.add(
        ids=batch_ids,
        embeddings=embeddings,
        documents=batch_texts,
        metadatas=batch_meta
    )

    if i % 320 == 0:
        print(f"📦 Procesados: {i}/{len(texts)} chunks...")

print(f"🏁 ¡CONSEGUIDO! Total en Chroma: {collection.count()}")

# PRUEBA RÁPIDA (SMOKE TEST)
query = "¿Quiénes son los padres de Jon Nieve según los rumores?"
q_emb = embedder.encode([query], normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print("\n🔍 PRUEBA DE BÚSQUEDA:")
for i, doc in enumerate(results['documents'][0]):
    print(f"[{i+1}] {doc[:200]}...")

### Web scrapping

De momento lo hacemos de personajes pero aquí se puede sacar de todo.

Metemos de momento 100 personajes a lo que ya hemos metido a la Chroma (en procesod e expansión).



In [ ]:
import requests
import re
import pandas as pd
import random

API_URL = "https://hieloyfuego.fandom.com/es/api.php"

# --- FILTROS DE RUIDO ---
EXCLUIR = ["Aegon", "Aenys", "Maegor", "Jaehaerys", "Viserys I", "Viserys II", "Daeron",
           "Baelor", "Maekar", "Aerys I", "Aerys II", "Rhaegar", "Lyanna", "Elia Martell",
           "Casa ", "Invernalia", "Desembarco", "Guardia", "Siete Reinos", "Apéndice", "Lugar"]

def obtener_nombres_apendice():
    print(" Conectando con la Ciudadela (Apéndice)...")
    params = {
        "action": "query", "format": "json", "titles": "Juego de Tronos-Apéndice",
        "prop": "links", "pllimit": "500"
    }
    res = requests.get(API_URL, params=params).json()
    pages = res.get('query', {}).get('pages', {})
    page_id = list(pages.keys())[0]
    links = [l['title'] for l in pages[page_id].get('links', [])]
    return [n for n in links if not any(m in n for m in EXCLUIR)]

def limpiar_texto(txt):
    # Eliminar plantillas {{...}} y tablas {|...|}
    txt = re.sub(r'\{\{.*?\}\}', '', txt, flags=re.DOTALL)
    txt = re.sub(r'\{\ land.*?\|\}', '', txt, flags=re.DOTALL)
    # Limpiar enlaces [[Archivo:...]] y [[Link|Texto]]
    txt = re.sub(r'\[\[(?:Archivo|Categoría|Imagen):.*?\]\]', '', txt, flags=re.IGNORECASE)
    txt = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]]*)\]\]', r'\1', txt)
    # Quitar negritas y saltos excesivos
    txt = re.sub(r"'''?", "", txt)
    return re.sub(r'\s+', ' ', txt).strip()

# --- PROCESAMIENTO ---
nombres_base = obtener_nombres_apendice()
random.shuffle(nombres_base)

datos_finales = []
LIMITE = 20

print(f"🎲 Analizando {len(nombres_base)} candidatos...")

for nombre in nombres_base:
    if len(datos_finales) >= LIMITE:
        break

    params = {
        "action": "query", "format": "json", "prop": "revisions",
        "titles": nombre, "rvprop": "content", "formatversion": "2"
    }

    try:
        res = requests.get(API_URL, params=params).json()
        p = res['query']['pages'][0]

        if "missing" in p: continue

        # Acceso directo al contenido según formatversion 2
        content = p['revisions'][0]['content']

        # CRITERIO: Debe ser un personaje (buscamos indicios de biografía)
        # Filtramos si es un lugar (contiene "asentamiento" o "castillo")
        es_lugar = any(x in content.lower() for x in ["asentamiento", "castillo", "fortaleza", "ciudad de"])
        es_historico = any(x in content.lower()[:500] for x in ["fue un rey", "antiguo rey", "fallecido en"])

        if not es_lugar and not es_historico:
            biografia = limpiar_texto(content)
            if len(biografia) > 200:
                datos_finales.append({"nombre": nombre, "biografia": biografia[:1200]})
                print(f"✅ [{len(datos_finales)}/{LIMITE}] {nombre}")

    except Exception:
        continue


if datos_finales:
    df = pd.DataFrame(datos_finales)
    df.to_csv("personajes_vivos_libro1.csv", index=False)
    print(f"\n✨ ¡Éxito! {len(df)} personajes listos.")
    print(df['nombre'].tolist())
else:
    print("\n❌ Sigue sin encontrar. Probablemente la estructura de la página ha cambiado.")

 Conectando con la Ciudadela (Apéndice)...
🎲 Analizando 258 candidatos...
✅ [1/20] Harwyn Hoare
✅ [2/20] Lysa Tully
✅ [3/20] El Rejo
✅ [4/20] Alerie Hightower
✅ [5/20] Tion Frey
✅ [6/20] Harwin
✅ [7/20] Arys Oakheart
✅ [8/20] Mikken
✅ [9/20] Dorna Swyft
✅ [10/20] Juego de Tronos-Capítulo 72
✅ [11/20] Forley Prester
✅ [12/20] Olenna Redwyne
✅ [13/20] Alannys Harlaw
✅ [14/20] Cerenna Lannister
✅ [15/20] Desmera Redwyne
✅ [16/20] Mar Angosto
✅ [17/20] Gerion Lannister
✅ [18/20] Aeron Greyjoy
✅ [19/20] Varly
✅ [20/20] Hobber Redwyne

✨ ¡Éxito! 20 personajes listos.
['Harwyn Hoare', 'Lysa Tully', 'El Rejo', 'Alerie Hightower', 'Tion Frey', 'Harwin', 'Arys Oakheart', 'Mikken', 'Dorna Swyft', 'Juego de Tronos-Capítulo 72', 'Forley Prester', 'Olenna Redwyne', 'Alannys Harlaw', 'Cerenna Lannister', 'Desmera Redwyne', 'Mar Angosto', 'Gerion Lannister', 'Aeron Greyjoy', 'Varly', 'Hobber Redwyne']


7, 8, 11, 12, 13, 17 no pertenecen al libro uno, así que se eliminan

In [ ]:
import uuid
import torch
import pandas as pd
import time

# --- 1) FILTRADO PREVIO (Lo que ya tenías) ---
indices_a_eliminar = [7, 8, 11, 12, 13, 17]
# Usamos el DataFrame que acabamos de scrapear con éxito
df_wiki_source = df.drop(indices_a_eliminar).reset_index(drop=True)

# --- 2) PROCESAMIENTO Y CHUNKING ---
wiki_records = []
print(f"🚀 Procesando {len(df_wiki_source)} personajes limpios de la Wiki...")

for i, row in df_wiki_source.iterrows():
    titulo = row['nombre']
    texto_completo = row['biografia']

    # Usamos tu función de chunk_text (asumiendo que está definida arriba)
    # Si no, una división simple por longitud funcionaría aquí
    chunks = chunk_text(texto_completo)

    for j, ch in enumerate(chunks):
        wiki_records.append({
            'chunk_id': f"wiki_{titulo.replace(' ', '_')}_{j}",
            'chapter_index': 999, # Marcador para diferenciar de los capítulos del libro
            'chapter_name': f"Wiki: {titulo}",
            'chunk_index': j,
            'text': ch,
            'source': 'fandom_wiki'
        })

    if i % 5 == 0: print(f"✅ Procesado: {titulo}...")

df_wiki_new = pd.DataFrame(wiki_records)

# --- 3) MERGE CON DATOS EXISTENTES ---
# Cargamos los chunks del Libro 1 que ya tienes procesados
path_antiguo = CHUNKS_DIR / 'chunks.parquet'
df_libros = pd.read_parquet(path_antiguo)
if 'source' not in df_libros.columns:
    df_libros['source'] = 'libro_1'

# Combinamos ambos mundos
df_total = pd.concat([df_libros, df_wiki_new], ignore_index=True)
path_nuevo = CHUNKS_DIR / 'chunks_combined.parquet'
df_total.to_parquet(path_nuevo, index=False)

print(f"\n📊 Dataset maestro creado: {len(df_total)} chunks totales.")

# --- 4) INYECCIÓN EN CHROMADB ---
print("🧠 Generando embeddings e inyectando en ChromaDB...")
prev_collection_count = collection.count()

if not df_wiki_new.empty:
    texts_to_embed = df_wiki_new['text'].tolist()
    # Metadatos: todo menos el texto
    metadatas_to_embed = df_wiki_new.drop(columns=['text']).to_dict(orient='records')
    # Generamos IDs únicos para Chroma
    ids_to_embed = [str(uuid.uuid4()) for _ in texts_to_embed]

    # Generación de embeddings con tu modelo de torch
    with torch.no_grad():
        embeddings = embedder.encode(texts_to_embed, normalize_embeddings=True)

    # Inyección masiva
    collection.add(
        ids=ids_to_embed,
        embeddings=embeddings.tolist(),
        documents=texts_to_embed,
        metadatas=metadatas_to_embed
    )

    actual_count = collection.count()
    new_records = actual_count - prev_collection_count
    print(f"🏁 Inyección completada. Nuevos registros: {new_records}")
    print(f"📚 Total actual en Chroma: {actual_count}")

    # --- 5) LIMPIEZA DE ARCHIVOS ---
    if path_nuevo.exists() and path_antiguo.exists() and new_records > 0:
        path_antiguo.unlink()
        print(f"🗑️ Archivo antiguo eliminado. El nuevo maestro es: 'chunks_combined.parquet'")
else:
    print("ℹ️ No se generaron chunks nuevos para procesar.")

display(df_wiki_new.head())

🚀 Procesando 14 personajes limpios de la Wiki...
✅ Procesado: Harwyn Hoare...
✅ Procesado: Harwin...
✅ Procesado: Mar Angosto...

📊 Dataset maestro creado: 2055 chunks totales.
🧠 Generando embeddings e inyectando en ChromaDB...
🏁 Inyección completada. Nuevos registros: 27
📚 Total actual en Chroma: 2055
🗑️ Archivo antiguo eliminado. El nuevo maestro es: 'chunks_combined.parquet'


,chunk_id,chapter_index,chapter_name,chunk_index,text,source
0,wiki_Harwyn_Hoare_0,999,Wiki: Harwyn Hoare,0,"El rey Harwyn Hoare, apodado Harwyn Manodura, ...",fandom_wiki
1,wiki_Harwyn_Hoare_1,999,Wiki: Harwyn Hoare,1,"os saqueadores en los Peldaños de Piedra, visi...",fandom_wiki
2,wiki_Lysa_Tully_0,999,Wiki: Lysa Tully,0,Lady Lysa Tully fue la segunda hija de Lord Ho...,fandom_wiki
3,wiki_Lysa_Tully_1,999,Wiki: Lysa Tully,1,aban el olor a leche agria.<ref>Juego de Trono...,fandom_wiki
4,wiki_El_Rejo_0,999,Wiki: El Rejo,0,El Rejo es una isla verde y fértil ubicada al ...,fandom_wiki


## Carga de la BBDD

Esto es para el notebook del RAG, quitar esta celda cuando esté metido alli

In [ ]:
import os
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import drive

# 1. Asegurar que Drive está montado
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. REDEFINIR RUTAS (Vital porque la memoria se borró)
PERSIST_BASE = Path('/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base')
CHROMA_DIR = PERSIST_BASE / 'chroma_db'
MODEL_NAME = 'BAAI/bge-large-en-v1.5' # El que usamos antes
COLLECTION_NAME = 'got_book1_chunks'

print(f"📂 Buscando base de datos en: {CHROMA_DIR}")

# 3. Conectar al cliente persistente
if CHROMA_DIR.exists():
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    try:
        # Intentamos pillar la colección
        collection = client.get_collection(name=COLLECTION_NAME)
        count = collection.count()
        if count > 0:
            print(f"✅ ¡Éxito! Conectado a '{COLLECTION_NAME}' con {count} chunks.")
        else:
            print("⚠️ La colección existe pero parece vacía.")
    except Exception as e:
        print(f"❌ Error al recuperar la colección: {e}")
        print("💡 Posibles colecciones encontradas:", client.list_collections())
else:
    print(f"❌ La ruta {CHROMA_DIR} no existe en Drive. Revisa la ruta manualmente.")

# 4. Cargar el modelo (necesario para procesar tus preguntas)
print(f"🧠 Cargando modelo de embeddings ({MODEL_NAME})...")
embedder = SentenceTransformer(MODEL_NAME, device="cuda")
print("🚀 Sistema listo para consultar.")

📂 Buscando base de datos en: /content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base/chroma_db
✅ ¡Éxito! Conectado a 'got_book1_chunks' con 2274 chunks.
🧠 Cargando modelo de embeddings (BAAI/bge-large-en-v1.5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🚀 Sistema listo para consultar.


## SMOKE TEST

Prueba del Retrieval sencillita

In [ ]:
# --- CONFIGURACIÓN DEL TEST ---
query = '¿Cómo escapó Tyrion del Nido de Águilas?'#'¿Cuál es el lema de la Casa Stark y qué significa para Ned su presencia en Invernalia?'

# 1. Generar embedding de la consulta con el modelo GTE (normalizado)
q_emb = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False).tolist()

# 2. Consultar Chroma
# Pedimos 5 resultados y que nos devuelva las 'distances' para validar el Cosine Similarity
n_results = 5
res = collection.query(
    query_embeddings=q_emb,
    n_results=n_results,
    include=['documents', 'metadatas', 'distances']
)

print(f"🔍 Consulta: '{query}'")
print(f"📊 Espacio de métrica: {collection.metadata.get('hnsw:space', 'L2 default')}")
print('=' * 80)

# 3. Mostrar resultados con Score de Similitud
if res['documents'][0]:
    for i in range(len(res['documents'][0])):
        doc = res['documents'][0][i]
        meta = res['metadatas'][0][i]
        dist = res['distances'][0][i]

        # En Cosine distance en Chroma: 0 es identidad, 1 es ortogonal.
        # Convertimos a Similitud (1 - distancia) para que sea más intuitivo
        similarity = 1 - dist

        print(f"📍 RESULTADO #{i+1} | Similitud: {similarity:.4f}")
        print(f"📖 Cap: {meta['chapter_index']} ({meta['chapter_name']}) | Chunk: {meta['chunk_index']}")
        print(f"📝 Texto: {doc[:400]}...")
        print('-' * 80)
else:
    print("❌ No se encontraron resultados. Revisa si la colección tiene datos.")

# 4. Verificación técnica de persistencia
print(f"📂 Verificando persistencia en Drive: {len(list(CHROMA_DIR.iterdir()))} archivos encontrados.")

🔍 Consulta: '¿Cómo escapó Tyrion del Nido de Águilas?'
📊 Espacio de métrica: cosine
📍 RESULTADO #1 | Similitud: 0.5455
📖 Cap: 41 (Text/045.xhtml) | Chunk: 4
📝 Texto: re fresco abundante, la luz del sol, y por las noches vea la luna y las estrellas, pero lo habra cambiado todo por el agujero ms sombro y hmedo de las entraas de Roca Casterly. Volars le haba asegurado Mord al empujarlo hacia el interior de la celda. Dentro de veinte das, o de treinta, o a lo mejor de cincuenta. Pero volars. Los Arryn contaban con la nica mazmorra de todo el reino en la que se per...
--------------------------------------------------------------------------------
📍 RESULTADO #2 | Similitud: 0.5444
📖 Cap: 41 (Text/045.xhtml) | Chunk: 2
📝 Texto: as de Cielo, al igual que les haba sucedido a tantos prisioneros del Nido de guilas a lo largo de los siglos. Bien pensado, no tengo tanta hambre declar mientras se retiraba al rincn de la celda. Mord gru, abri los dedos, y el viento se llev el plato. Unas cuantas al